# ADS v2: SCV attachment

Attaches an SCV_Customer_Key to every event in ads_v2.EventStream. Cascade, first match wins (ScvLinkedBy says which): policy code, travel customer ('M' + Mapfre customer id), quote reference in its own quote system, quote reference in any system, client code (given, or the policy's client code, or the first six characters of the policy code as notebook 05 cell 19), then the contact (email / phone hash key) as at the event time through ContactScvLink. Where a rule finds several SCV keys the lowest is kept (05 cell 22) and ScvCandidates says how many. Unlinked events keep a null key (no negative row-number keys) with an ExceptionReason, and every unlinked reference is listed once in ScvLinkExceptions for follow-up with the client.

Generated by `ads_build/v2/build_ads.py` from `families/*.yaml`. Event descriptions come from `seeds/ChillEpisodeTypes.csv`.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ads_build/v2/families/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run ADS_v2_00_Setup

In [ ]:
%%pyspark
ads_run_log("20_ScvAttach", "started")


In [ ]:
/******************************************************************************************************/
/*  Name        : ads_v2.ContactScvLink                                                                 */
/*  Description : The contact link built in ADS_v2_10_Common (stg.adsv2_contact_scv_link, used there fo */
/*              : r call attribution), published: email / phone hash key -> SCV key with its validity w */
/*              : indow [ValidFrom, ValidTo).                                                           */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_contact_scv_link                                                     */
/******************************************************************************************************/
create or replace table ${ads.target}.ContactScvLink as
select  *
from    ${ads.stg}.adsv2_contact_scv_link

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_event_ref                                                                   */
/*  Description : Each event's references in link form: policy code (upper), quote reference (lower) an */
/*              : d its quote system (from the event's QuoteId, else the event's own source system when */
/*              :  that is a quote system), client code (given, else the policy's client code, else the */
/*              :  first six characters of the policy code), travel customer ('M' + Mapfre id) and cont */
/*              : act (the event's EmailKey / PhoneKey, or the call's customer PhoneKey).               */
/*              :                                                                                       */
/*              : Reads: ads_v2.EventStream, ods_v2.Quote, ods_v2.Policy, stg.adsv2_call_contact        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_event_ref as
select  e.EventId,
        e.EventDateTime,
        nullif(upper(trim(e.SourcePolicyReference)), '')                                   as PolicyRef,
        nullif(lower(trim(e.SourceQuoteReference)), '')                                    as QuoteRef,
        coalesce(q.SourceSystemId,
                 case when e.SourceSystemId in (2, 3, 4, 7, 8, 9) then e.SourceSystemId end) as QuoteSystemId,
        case when q.PolicyId is not null then 1 else 0 end              as QuoteSold,
        case when e.SourceCustomerReferenceType = 'ClientCode' then nullif(upper(trim(e.SourceCustomerReference)), '')
             else coalesce(p.ClientCode, left(nullif(upper(trim(e.SourcePolicyReference)), ''), 6)) end as ClientRef,
        case when e.SourceCustomerReferenceType = 'ClientCode' then 'client'
             when p.ClientCode is not null then 'client (policy''s client)'
             when e.SourcePolicyReference is not null then 'client (policy code prefix)' end as ClientRefFrom,
        case when e.SourceCustomerReferenceType = 'MapfreCustomerId'
             then concat('M', nullif(upper(trim(e.SourceCustomerReference)), '')) end      as TravelRef,
        case when e.SourceCustomerReferenceType in ('EmailKey', 'PhoneKey') then e.SourceCustomerReference
             else cc.PhoneKey end                                       as ContactKey,
        case when e.SourceCustomerReferenceType in ('EmailKey', 'PhoneKey') then e.SourceCustomerReferenceType
             when cc.PhoneKey is not null then 'PhoneKey' end           as ContactKeyType,
        case when e.SourceCustomerReferenceType = 'UserIdKey' then e.SourceCustomerReference end as UserIdRef
from    ${ads.target}.EventStream e
left join ${ads.ods}.Quote q            on q.QuoteId = e.QuoteId
left join ${ads.ods}.Policy p           on p.PolicyCode = nullif(upper(trim(e.SourcePolicyReference)), '')
left join ${ads.stg}.adsv2_call_contact cc
        on  e.SourceSystemId = 20 and cc.ConversationId = e.EventSourceReference

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_event_scv                                                                   */
/*  Description : Each event's link by every rule (null where the rule finds nothing), the contact as a */
/*              : t the event time (the ContactScvLink window holding the event) and whether the contac */
/*              : t is known at all.                                                                    */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_event_ref, stg.adsv2_scv_by_policy, stg.adsv2_scv_by_customer, stg.a */
/*              : dsv2_scv_by_quote, ads_v2.ContactScvLink                                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_event_scv as
select  r.*,
        pl.SCV_Customer_Key                                             as PolicyScv,
        pl.ScvCandidates                                                as PolicyScvCandidates,
        tr.SCV_Customer_Key                                             as TravelScv,
        tr.ScvCandidates                                                as TravelScvCandidates,
        qs.SCV_Customer_Key                                             as QuoteScv,
        qs.ScvCandidates                                                as QuoteScvCandidates,
        qa.SCV_Customer_Key                                             as QuoteAnyScv,
        qa.ScvCandidates                                                as QuoteAnyScvCandidates,
        cl.SCV_Customer_Key                                             as ClientScv,
        cl.ScvCandidates                                                as ClientScvCandidates,
        ck.SCV_Customer_Key                                             as ContactScv,
        ck.ScvCandidates                                                as ContactScvCandidates,
        ck.LinkBasis                                                    as ContactLinkBasis,
        case when kn.ContactKey is not null then 1 else 0 end           as ContactKnown
from    ${ads.stg}.adsv2_event_ref r
left join ${ads.stg}.adsv2_scv_by_policy pl
        on  pl.PolicyRef = r.PolicyRef
left join ${ads.stg}.adsv2_scv_by_customer tr
        on  tr.SourceSystemId = 5 and tr.CustomerRef = r.TravelRef
left join ${ads.stg}.adsv2_scv_by_quote qs
        on  qs.QuoteRef = r.QuoteRef and qs.SourceSystemId = r.QuoteSystemId
left join (
        select  QuoteRef, min(SCV_Customer_Key) as SCV_Customer_Key, count(distinct SCV_Customer_Key) as ScvCandidates
        from    ${ads.stg}.adsv2_scv_by_quote
        group by QuoteRef
        ) qa    on  qa.QuoteRef = r.QuoteRef
left join ${ads.stg}.adsv2_scv_by_customer cl
        on  cl.SourceSystemId = 1 and cl.CustomerRef = r.ClientRef
left join ${ads.target}.ContactScvLink ck
        on  ck.ContactKey = r.ContactKey and ck.ContactKeyType = r.ContactKeyType
        and r.EventDateTime >= ck.ValidFrom and (ck.ValidTo is null or r.EventDateTime < ck.ValidTo)
left join (
        select  distinct ContactKey, ContactKeyType
        from    ${ads.target}.ContactScvLink
        ) kn    on  kn.ContactKey = r.ContactKey and kn.ContactKeyType = r.ContactKeyType

In [ ]:
/******************************************************************************************************/
/*  Name        : ads_v2.ScvEventStream                                                                 */
/*  Description : One row per event with its SCV key, how it was linked (ScvLinkedBy), how many SCV key */
/*              : s the winning rule found (ScvCandidates) and, when unlinked, the reason and the refer */
/*              : ence to follow up (priority 1 = policy, client, travel customer or a quote that becam */
/*              : e a policy; 2 = other quotes; 3 = contact or nothing).                                */
/*              :                                                                                       */
/*              : Reads: ads_v2.EventStream, stg.adsv2_event_scv                                        */
/******************************************************************************************************/
create or replace table ${ads.target}.ScvEventStream as
select  e.*,
        coalesce(s.PolicyScv, s.TravelScv, s.QuoteScv, s.QuoteAnyScv, s.ClientScv, s.ContactScv) as SCV_Customer_Key,
        case when s.PolicyScv is not null then 'policy'
             when s.TravelScv is not null then 'travel customer'
             when s.QuoteScv is not null then 'quote'
             when s.QuoteAnyScv is not null then 'quote (other system)'
             when s.ClientScv is not null then s.ClientRefFrom
             when s.ContactScv is not null and s.ContactLinkBasis = 'look-ahead' then 'contact (look-ahead)'
             when s.ContactScv is not null then 'contact'
             else 'none' end                                            as ScvLinkedBy,
        coalesce(case when s.PolicyScv is not null then s.PolicyScvCandidates end,
                 case when s.TravelScv is not null then s.TravelScvCandidates end,
                 case when s.QuoteScv is not null then s.QuoteScvCandidates end,
                 case when s.QuoteAnyScv is not null then s.QuoteAnyScvCandidates end,
                 case when s.ClientScv is not null then s.ClientScvCandidates end,
                 case when s.ContactScv is not null then s.ContactScvCandidates end, 0) as ScvCandidates,
        case when coalesce(s.PolicyScv, s.TravelScv, s.QuoteScv, s.QuoteAnyScv, s.ClientScv, s.ContactScv) is not null then null
             when s.PolicyRef is not null then 'policy not in scveadmlinkfinal'
             when s.QuoteRef is not null
             then concat('quote not in scveadmlinkfinal (system ', coalesce(cast(s.QuoteSystemId as string), 'unknown'), ')')
             when s.TravelRef is not null then 'travel customer not in scveadmlinkfinal'
             when s.ClientRef is not null then 'client not in scveadmlinkfinal'
             when s.ContactKey is not null and s.ContactKnown = 0 then 'contact not found'
             when s.ContactKey is not null then 'contact not used near event time'
             when s.UserIdRef is not null then 'user id only (not linkable)'
             else 'no reference' end                                    as ExceptionReason,
        case when coalesce(s.PolicyScv, s.TravelScv, s.QuoteScv, s.QuoteAnyScv, s.ClientScv, s.ContactScv) is not null then null
             when s.PolicyRef is not null then 'PolicyCode'
             when s.QuoteRef is not null then 'QuoteReference'
             when s.TravelRef is not null then 'TravelCustomer'
             when s.ClientRef is not null then 'ClientCode'
             when s.ContactKey is not null then s.ContactKeyType
             when s.UserIdRef is not null then 'UserIdKey'
             else 'none' end                                            as ExceptionReferenceType,
        case when coalesce(s.PolicyScv, s.TravelScv, s.QuoteScv, s.QuoteAnyScv, s.ClientScv, s.ContactScv) is not null then null
             else coalesce(s.PolicyRef, s.QuoteRef, s.TravelRef, s.ClientRef, s.ContactKey, s.UserIdRef) end as ExceptionReference,
        case when coalesce(s.PolicyScv, s.TravelScv, s.QuoteScv, s.QuoteAnyScv, s.ClientScv, s.ContactScv) is not null then null
             when s.PolicyRef is not null or s.TravelRef is not null or s.ClientRef is not null then 1
             when s.QuoteRef is not null and s.QuoteSold = 1 then 1
             when s.QuoteRef is not null then 2
             else 3 end                                                 as ExceptionPriority
from    ${ads.target}.EventStream e
join    ${ads.stg}.adsv2_event_scv s  on s.EventId = e.EventId

In [ ]:
%%pyspark
# ads_v2.ScvLinkExceptions: One row per distinct unlinked reference (policy / quote / client code as held, contact as a hash key) with its reason, event count, first / last event date, event types and priority. FirstSeenBatch is kept from the previous run's table, so a follow-up list can show what is new.
#
# Reads: ads_v2.ScvEventStream, ads_v2.ScvLinkExceptions
from pyspark.sql import functions as F
tbl = f"{target_schema}.ScvLinkExceptions"
keys = ["ReferenceType", "Reference", "SourceSystemId", "PolicyTypeGroup", "ExceptionReason"]
ex = spark.sql(f"""
    select  ExceptionReferenceType as ReferenceType, ExceptionReference as Reference, SourceSystemId,
            PolicyTypeGroup, ExceptionReason,
            count(*) as EventCount, min(EventDate) as FirstEventDate, max(EventDate) as LastEventDate,
            concat_ws(', ', sort_array(collect_set(EventTypeId))) as EventTypes,
            min(ExceptionPriority) as Priority
    from    {target_schema}.ScvEventStream
    where   SCV_Customer_Key is null
    group by ExceptionReferenceType, ExceptionReference, SourceSystemId, PolicyTypeGroup, ExceptionReason""")
if spark.catalog.tableExists(tbl):
    prev = spark.table(tbl).select(*keys, F.col("FirstSeenBatch").alias("PrevFirstSeen"))
    cond = [ex[k].eqNullSafe(prev[k]) for k in keys]
    ex = ex.join(prev, cond, "left").select(ex["*"], prev["PrevFirstSeen"])
else:
    ex = ex.withColumn("PrevFirstSeen", F.lit(None).cast("string"))
ex = (ex.withColumn("FirstSeenBatch", F.coalesce("PrevFirstSeen", F.lit(batch_id)))
        .withColumn("LastSeenBatch", F.lit(batch_id)).drop("PrevFirstSeen"))
# written to stg first: a table cannot be overwritten in the same plan that reads it
ex.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{stg_schema}.adsv2_scv_exceptions_new")
(spark.table(f"{stg_schema}.adsv2_scv_exceptions_new")
      .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl))


In [ ]:
%%pyspark
# link rate: Link rate by product and rule, and the exceptions by reason (printed; the tables hold the detail).
#
# Reads: ads_v2.ScvEventStream, ads_v2.ScvLinkExceptions
spark.sql(f"""
    select  PolicyTypeGroup, ScvLinkedBy, count(*) as Events,
            round(100 * count(*) / sum(count(*)) over (partition by PolicyTypeGroup), 1) as PctOfProduct,
            sum(case when ScvCandidates > 1 then 1 else 0 end) as SeveralScvKeys
    from    {target_schema}.ScvEventStream
    group by PolicyTypeGroup, ScvLinkedBy
    order by PolicyTypeGroup, Events desc""").show(100, truncate=False)
spark.sql(f"""
    select  ExceptionReason, Priority, count(*) as References, sum(EventCount) as Events
    from    {target_schema}.ScvLinkExceptions
    group by ExceptionReason, Priority
    order by Events desc""").show(100, truncate=False)


## Checks


In [ ]:
%%pyspark
# Checks for the tables this notebook built. All are checked and logged to AdsBuildLog; the cell then fails if
# any check failed, naming the table and the likely cause.
import json, re
from functools import reduce
from pyspark.sql import functions as F
SPEC = json.loads(r"""[{"table": "ContactScvLink", "bk": ["ContactKey", "ContactKeyType", "ValidFrom"], "sk": null, "nullable": [], "allow": []}, {"table": "ScvEventStream", "bk": ["EventId"], "sk": null, "nullable": [], "allow": []}, {"table": "ScvLinkExceptions", "bk": ["ReferenceType", "Reference", "SourceSystemId", "PolicyTypeGroup", "ExceptionReason"], "sk": null, "nullable": ["Reference", "SourceSystemId"], "allow": []}]""")
PII_NAME = re.compile(r"(e_?mail|forename|first_?name|surname|last_?name|full_?name|second_?name|phone|mobile|telephone|post_?code|eircode(?!_?area)|address_?line|risk_?address|postal_?address|dob$|date_?of_?birth|licen[cs]e_?num|registration_?no|reg_?number|registration_number|^ani$|^dnis$|originatingdnis|^payer_|iban|^bic|^contact$|rendered_?message)", re.I)
EMAIL_RE = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
MOBILE_RE = r"(^|[^0-9A-Za-z-])(\+?353 ?|0)8[35-9][ -]?[0-9]{3}[ -]?[0-9]{4}([^0-9A-Za-z-]|$)"   # stands alone, not inside an id
rows, failures = [], []
for s in SPEC:
    t = f"{target_schema}.{s['table']}"
    df = spark.table(t)
    bk = s["bk"]
    must = [c for c in bk if c not in s["nullable"]]
    agg = df.agg(F.count(F.lit(1)).alias("n"),
                 F.countDistinct(F.struct(*bk)).alias("d_bk"),
                 F.sum(F.when(reduce(lambda a, b: a | b, [F.col(c).isNull() for c in must] or [F.lit(False)]), 1).otherwise(0)).alias("null_bk"),
                 *( [F.countDistinct(s["sk"]).alias("d_sk")] if s["sk"] else [] )).first()
    pii_cols = [c for c in df.columns if PII_NAME.search(c) and not c.endswith("Key") and c not in s["allow"]]
    text_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]   # hashes never match
    hits = {}
    if text_cols:
        found = df.select(*text_cols).limit(1000000).agg(
            *[F.sum((F.col(c).rlike(EMAIL_RE) | F.col(c).rlike(MOBILE_RE)).cast("int")).alias(c) for c in text_cols]).first().asDict()
        hits = {c: v for c, v in found.items() if v}
    problems, advice = [], []
    keys = " + ".join(bk)
    if agg["n"] == 0:
        problems.append("empty")
        advice.append("the table is empty: its source is empty, or a filter (cut-off date, join) removed every row")
    if agg["d_bk"] != agg["n"]:
        problems.append(f"{agg['n'] - agg['d_bk']:,} duplicate business keys")
        advice.append(f"several rows share a key ({keys}): the source sent the same record more than once, or the "
                      f"dedupe rule in the cell no longer holds. Examples: select {', '.join(bk)}, count(*) from {t} "
                      f"group by {', '.join(bk)} having count(*) > 1")
    if agg["null_bk"]:
        problems.append(f"{agg['null_bk']:,} null business keys")
        advice.append(f"key columns ({', '.join(must)}) are empty on some rows: if that is legitimate for a column, add it "
                      f"to key_nullable for this table in the config; otherwise the source changed")
    if s["sk"] and agg["d_sk"] != agg["n"]:
        problems.append("surrogate key not unique")
        advice.append("the surrogate key repeats: the business key is not unique (see above)")
    if pii_cols:
        problems.append(f"PII-named columns {pii_cols}")
        advice.append("columns named like personal data: hash them (HASH_*) or leave them out in the config")
    if hits:
        problems.append(f"email/mobile values in {hits}")
        advice.append("emails / mobile numbers inside these columns: wrap the column in REDACT() (or hash it) in the config")
    rows.append((batch_id, s["table"], agg["n"], agg["n"] - agg["d_bk"], agg["null_bk"], "; ".join(problems) or "ok"))
    print(f"{s['table']:<28} {agg['n']:>13,}  {'; '.join(problems) or 'ok'}")
    if problems:
        failures.append(f"{s['table']}: " + " | ".join(advice))
log = spark.createDataFrame(rows, "AdsBatchId string, TableName string, RowCount long, DuplicateKeys long, NullKeys long, Result string")
log.withColumn("CheckedAt", F.current_timestamp()).write.mode("append").saveAsTable(f"{target_schema}.AdsBuildLog")
if failures:
    ads_run_log("20_ScvAttach", "failed", "; ".join(f.split(":")[0] for f in failures))
    raise AssertionError("ads checks failed:\n- " + "\n- ".join(failures))
ads_run_log("20_ScvAttach", "succeeded")


In [ ]:
%%pyspark
ads_run_log("20_ScvAttach", "succeeded")
